## Part 1: NumPy Practical Lab

### Task 1.1: Environmental Sensor Analytics

A sensor records hourly temperatures over 72 hours, with a few corrupted readings injected.

In [1]:
import numpy as np

np.random.seed(42)

raw_temperatures = np.random.normal(loc=24.0, scale=3.0, size=72)

# Simulated corrupt readings
raw_temperatures[[5, 12, 28, 45, 60]] = [99.9, -15.0, 88.8, -50.0, 102.4]

print("Raw temperatures (first 10):", np.round(raw_temperatures[:10], 2))

Raw temperatures (first 10): [25.49 23.59 25.94 28.57 23.3  99.9  28.74 26.3  22.59 25.63]


**Step 1 — Boolean mask for valid readings**

A reading is valid if it falls within the realistic range 15.0 °C to 40.0 °C (inclusive).

In [2]:
# Boolean mask: True where the reading is physically realistic
valid_mask = (raw_temperatures >= 15.0) & (raw_temperatures <= 40.0)

print("Valid mask (first 10):", valid_mask[:10])
print("Number of invalid (corrupted) readings:", np.sum(~valid_mask))

Valid mask (first 10): [ True  True  True  True  True False  True  True  True  True]
Number of invalid (corrupted) readings: 5


**Step 2 — Replace invalid values with the median of the valid readings**

In [3]:
# Compute the median using ONLY the valid readings
median_valid_temp = np.median(raw_temperatures[valid_mask])

# Work on a copy so the original corrupted array is preserved for reference
cleaned_temperatures = raw_temperatures.copy()

# Replace every invalid entry with the valid-only median (vectorized, no loop)
cleaned_temperatures[~valid_mask] = median_valid_temp

print("Median used for replacement:", round(median_valid_temp, 2))
print("Cleaned temperatures (first 10):", np.round(cleaned_temperatures[:10], 2))

Median used for replacement: 23.65
Cleaned temperatures (first 10): [25.49 23.59 25.94 28.57 23.3  23.65 28.74 26.3  22.59 25.63]


**Step 3 — Summary statistics on the cleaned data**

In [4]:
mean_temp = round(np.mean(cleaned_temperatures), 2)
std_temp = round(np.std(cleaned_temperatures), 2)
var_temp = round(np.var(cleaned_temperatures), 2)

print(f"Mean:               {mean_temp}")
print(f"Standard Deviation: {std_temp}")
print(f"Variance:           {var_temp}")

Mean:               23.74
Standard Deviation: 2.69
Variance:           7.24


### Task 1.2: Matrix Slicing & Strided Operations

In [5]:
grid = np.arange(10, 74).reshape(8, 8)
print("Original 8x8 grid:")
print(grid)

Original 8x8 grid:
[[10 11 12 13 14 15 16 17]
 [18 19 20 21 22 23 24 25]
 [26 27 28 29 30 31 32 33]
 [34 35 36 37 38 39 40 41]
 [42 43 44 45 46 47 48 49]
 [50 51 52 53 54 55 56 57]
 [58 59 60 61 62 63 64 65]
 [66 67 68 69 70 71 72 73]]


**1. Extract the center 4x4 portion**

In [6]:
# Rows/cols 2 through 5 (inclusive) sit exactly in the middle of an 8x8 grid
center_4x4 = grid[2:6, 2:6]
print("Center 4x4 block:")
print(center_4x4)

Center 4x4 block:
[[28 29 30 31]
 [36 37 38 39]
 [44 45 46 47]
 [52 53 54 55]]


**2. Extract the last column using a single slicing statement**

In [7]:
last_column = grid[:, -1]
print("Last column:", last_column)

Last column: [17 25 33 41 49 57 65 73]


**3. Downsample by taking every second row and every second column**

In [8]:
# Step-2 slicing on both axes halves each dimension: 8x8 to 4x4
downsampled = grid[::2, ::2]
print("Downsampled 4x4 matrix:")
print(downsampled)
print("Shape:", downsampled.shape)

Downsampled 4x4 matrix:
[[10 12 14 16]
 [26 28 30 32]
 [42 44 46 48]
 [58 60 62 64]]
Shape: (4, 4)


## Part 2: Sliding Median Filter

A noisy signal contains random spikes. We'll smooth it out using a size 3 sliding median filter.

In [9]:
signal = np.array([10, 11, 99, 12, 13, -45, 14, 15, 16])

# Work on a copy so the original noisy signal stays intact for comparison
filtered_signal = signal.copy()

# Slide a window of size 3 across every interior index (endpoints have no
# left/right neighbor to form a full window, so they are left unchanged)
for index in range(1, len(signal) - 1):
    window = signal[index - 1: index + 2]      # 3-element window via slicing
    filtered_signal[index] = np.median(window)  # replace with the window's median

print("Original signal: ", signal)
print("Filtered signal: ", filtered_signal)

Original signal:  [ 10  11  99  12  13 -45  14  15  16]
Filtered signal:  [10 11 12 13 12 13 14 15 16]


**Explanation**

The median filter removes the spikes at `99` and `-45` because a median only reflects the
*middle* value of a small, sorted window it isn't dragged around by one extreme outlier the
way a sum-based average is. For example, the window `[11, 99, 12]` sorts to `[11, 12, 99]`,
and the median (`12`) simply ignores the spike entirely, since `99` is just the largest of
three values rather than something that shifts the "center." The same happens with `-45`:
the window `[13, -45, 14]` sorts to `[-45, 13, 14]`, and the median (`13`) is unaffected by
how far below the rest of the data that spike sits.

A **mean** filter would perform far worse here, because every value in the window including
the corrupted one contributes proportionally to the result. Averaging `[11, 99, 12]` gives
`40.67`, which is nowhere close to the true signal level (~11–12) and effectively smears the
spike's error across neighboring points instead of removing it. The median is robust to this
kind of extreme, sparse noise precisely because it depends only on rank/order rather than
magnitude, so a single wild outlier can shift a mean drastically but can barely move a median
at all.